In [2]:
import pandas as pd
import numpy as np
# import spacy
# import scispacy
# import spacy
# import re
# from spacy.pipeline import EntityRuler
# nlp = spacy.load("en_ner_bionlp13cg_md")
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC


In [3]:
## Loading the dfs
df = pd.read_csv("everything.csv", index_col=0)
df_doc = pd.read_csv('specialists.csv')

In [4]:
# Categorize.  Set dictionaries
col = ['label', 'anat_symp_lem', 'symp_lem']
df = df[col]
df = df[(pd.notnull(df['anat_symp_lem'])) & (pd.notnull(df['symp_lem']))]
df['category_id'] = df['label'].factorize()[0]
category_id_df = df[['label', 'category_id']].drop_duplicates().sort_values('category_id')
category_to_id = dict(category_id_df.values)
id_to_category = dict(category_id_df[['category_id', 'label']].values)
df

,label,anat_symp_lem,symp_lem,category_id
0,"Ear, Nose, Throat",sore gland otitis externa pus-like ...,ear pain itching irritation around ear can...,0
1,"Ear, Nose, Throat",mucus chest eye nose earache ...,frequent sneeze runny block nose itchy r...,0
2,"Ear, Nose, Throat",taste eye muscle muscle ear e...,sore throat block runny nose sneeze cough ho...,0
3,"Ear, Nose, Throat",ear ménière vertigo spin tinnitus hear lo...,vertigo - sensation environment around move ...,0
4,"Ear, Nose, Throat",eye brain leg skull body brui...,mild headache nausea feel sick mild dizzin...,0
...,...,...,...,...
975,"Ear, Nose, Throat",face saliva eye,suddenly face could not move right saliva drip...,0
976,Neurologist,stroke,grandfather stroke no long walk foot,1
977,Neurologist,,cousin s head turn motor collide lose consci...,1
978,Neurologist,fall,feel like fall right walk,1


In [18]:
# X_train, X_test, y_train, y_test = train_test_split(df['anat_symp_lem'], df['label'], random_state = 42)
# count_vect = CountVectorizer()
# X_train_counts = count_vect.fit_transform(X_train)
# tfidf_transformer = TfidfTransformer()
# X_train_tfidf = tfidf_transformer.fit_transform(X_train_counts)
# clf = LogisticRegression(penalty='l2', dual=False, tol=0.0001, C=1.0, fit_intercept=True, intercept_scaling=1, class_weight=None, random_state=42, solver='newton-cg', max_iter=100, multi_class='auto', verbose=0, warm_start=False, n_jobs=None, l1_ratio=None).fit(X_train_tfidf, y_train)

--------------------------------------------------
  Accuracy: 92.653%
       F1 : 92.556%
--------------------------------------------------


In [6]:
df_doc

,Doctor,Specialty,Location,Clinic,Contact,Online
0,"Criscely L. Go, MD",Neurologist,Manila,"Metropolitan Medical Center Room 224, Masangka...",(02) 425 9779 / (02) 254 1111 local 2033 / (09...,NaN
1,"Dominador D. Baldonado, MD",Neurologist,Quezon City,FEU-NRMF Medical Center Room 415 Marian Medica...,(02) 461 3134 / (02) 427 0213,NaN
2,"Euvin Paul G. Lagapa, MD",Neurologist,Davao City,"San Pedro Hospital Room 307 Rita Stang Bldg., ...",(082) 221 7272,NaN
3,"Paul Matthew D. Pasco, MD",Neurologist,Manila,"Philippine General Hospital, Taft Avenue, Ermi...",(02) 8521 8450,NaN
4,"Abdias V. Aquino, MD",Neurologist,Quezon City,"National Kidney and Transplant Institute, East...",(02) 8981 0400,NaN
...,...,...,...,...,...,...
174,"Flores, Ruben",Neurologist,Iligan,NaN,drrubenflores.neurology@gmail.com,NaN
175,"Diamla, Sharimah",Neurologist,Iligan/Marawi,NaN,shahjamla@yahoo.com,NaN
176,"Ugdamina, Fritzie",Neurologist,Ozamiz City,NaN,junfritz7583@yahoo.com,NaN
177,"Yap-Domingo, Archie",Neurologist,Ozamiz City,NaN,archieyapdomingo@gmail.com,NaN


In [30]:
# USER INPUT
text = "my eyes are red and itchy. it is swollen and it's painful to touch .it's been reddish since last week."

location = 'Manila'

In [31]:
# Predict if len > 70
if len(text[0])<=70:
    print('Tell me more')

else:
    if max(clf.predict_proba((count_vect.transform([text])))[0]) > 0.50:
        doc = ''.join(list(clf.predict(count_vect.transform(text))))
        print(doc)
    else:
        print("""I cannot recommend a specialist based on your symptoms.  
                 I suggest that you seek consult with a General Physician via telemedicine.
                 Here are the recommendations:
                 
                 Doctor's Name: Dr. Maria Bathilda I. Tampi
                 Telemedicine Link: drmabait@consult.com
                 
                 Doctor's Name: Dr. Leopold G.  Hiran
                 Telemedicine Link: nicedoctor@telemed.com
                    
              """)
text[0]

Tell me more


'm'

In [32]:
# Add the next text to the original text

if len(text)<70:
    print('Tell me more')

else:
    if max(clf.predict_proba((count_vect.transform([text])))[0]) > 0.50:
        doc = ''.join(list(clf.predict(count_vect.transform([text]))))
        print(doc)
    else:
        print("""I cannot recommend a specialist based on your symptoms.  
                 I suggest that you seek consult with a General Physician via telemedicine.
                 Here are the recommendations:
                 
                 Doctor's Name: Dr. Maria Bathilda I. Tampi
                 Telemedicine Link: drmabait@consult.com
                 
                 Doctor's Name: Dr. Leopold G.  Hiran
                 Telemedicine Link: nicedoctor@telemed.com
                    
              """)
clf.predict_proba((count_vect.transform([text])))[0]

Ophthalmologist


array([0.43171569, 0.06658065, 0.50170366])

In [188]:
# Results
df_res = df_doc[(df_doc['Specialty ']==doc) & (df_doc['Location'].str.contains(location))]
df_online = df_doc.replace(np.nan, '', regex=True)
df_online = df_online[(df_online['Specialty ']==doc) & (df_online['Online']!="")]            

In [189]:
# Print results
if len(df_res) == 0:
    print(f"""      Searched for: {doc} in {location}
      None found in the list
      
      Another option is consulting via telemedicine with the following specialists:
      """)
    for index,row in df_online.iterrows():
        print(f"""
            Doctor's Name: {row["Doctor"]}
            Link for Telemedicine Consult: {row["Online"]}
        """)
else:
    print(f"       Searched for: {doc} in {location}")
    for index,row in df_res.iterrows():
        print(f"""
            Doctor's Name: {row["Doctor"]}
            Clinic Address: {(row["Clinic"])}
            Contact Details For Appointment: {row["Contact"]}
        """)

       Searched for: Ear, Nose, Throat in Manila

            Doctor's Name: Dr. Elmer M. Dela Cruz
            Clinic Address: Room 809, Medical Arts Center 1, Manila Doctors Hospital, United Nations Avenue, Ermita, 1000, Manila,
            Contact Details For Appointment: (02) 8523 9406
        


In [10]:
sample = ['My ears hurt since last week.  I put baby oil because I can feel an insect moving inside but the pain remains',
         'I scratched my itchy eyes yesterday.  Now, I cannot see clearly. Please help', 'My son has fever for a day.  I think his head is small compared to other babies.', ]

In [11]:
locs = ['Quezon City', 'Davao', 'Sulu']

# Saving the models

In [12]:
import joblib

In [13]:
# LR model
joblib_file = "joblib_LR_Model.pkl"  
joblib.dump(clf, joblib_file)

['joblib_LR_Model.pkl']

In [14]:
joblib_LR_model = joblib.load(joblib_file)
joblib_LR_model

LogisticRegression(random_state=42, solver='newton-cg')

In [ ]:
# tfidf
tfidf_transformer = TfidfTransformer()
joblib_cv = "joblib_tfidf.pkl"  
joblib.dump(tfidf_transformer, joblib_cv)


